# Ajuste y validación de modelos de aprendizaje automático

El ajuste usa exclusivamente la partición `train` archivada. Se compara la logística M5 con árbol, bosque y boosting. La búsqueda estratificada de cinco pliegues selecciona por ROC-AUC. PR-AUC se calcula como average precision (AP). La evaluación externa usa `val` y umbral 0.50.

In [ ]:
from importlib.metadata import version
from importlib.util import find_spec
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile
import json
import sys

root = Path.cwd()
if not (root / "src" / "preprocessing.py").is_file():
    root = root.parent
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display
from threadpoolctl import threadpool_limits

from src.evaluation import (
    compare_feature_importance, probability_metrics, summarize_search, threshold_metrics,
)
from src.models import fit_phase4_logistic, load_model, optimize_model, save_model
from src.preprocessing import load_and_split

boosting = "xgboost" if find_spec("xgboost") is not None else "hist_gradient_boosting"
model_names = {
    "logistic": "Logística M5",
    "tree": "Árbol",
    "forest": "Random Forest",
    boosting: "XGBoost" if boosting == "xgboost" else "HistGradientBoosting",
}
archive_path = root / "sources" / "phase1_deliverables.zip"
splits = load_and_split(archive_path)
X_train, y_train = splits["train"]
X_val, y_val = splits["val"]
del splits
with ZipFile(archive_path) as archive:
    with np.load(BytesIO(archive.read("data/phase1/split_indices.npz"))) as archived:
        np.testing.assert_array_equal(X_train.index, archived["train"])
        np.testing.assert_array_equal(X_val.index, archived["validation"])

partitions = {
    "train": {"n": len(y_train), "defaults": int(y_train.sum())},
    "val": {"n": len(y_val), "defaults": int(y_val.sum())},
}
print(model_names[boosting], partitions)


## Ajuste sobre entrenamiento

La logística evalúa la especificación M5. Los modelos conservan la distribución observada de clases. Se reporta la media y desviación estándar poblacional de cada métrica entre pliegues.

In [ ]:
searches = {}
cv = {}
for name, label in model_names.items():
    searches[name] = optimize_model(name, X_train, y_train)
    cv[name] = summarize_search(searches[name])
    print(label, cv[name]["candidatos"], cv[name]["parámetros"], flush=True)

cv_table = pd.DataFrame([
    {"modelo": model_names[name], "métrica": metric, "conjunto": split, **stats}
    for name, summary in cv.items()
    for metric, groups in summary["métricas"].items()
    for split, stats in groups.items()
]).drop(columns="pliegues")
display(cv_table.set_index(["modelo", "métrica", "conjunto"]).unstack())


## Evaluación en validación

La columna «train completo» usa el modelo reajustado con las 21.000 observaciones. Las métricas de clasificación en validación usan PD ≥ 0.50.

In [ ]:
reports = root / "reports"
reports.mkdir(exist_ok=True)
validation = {}
full_train = {}
with threadpool_limits(1):
    baseline = fit_phase4_logistic(X_train, y_train).predict_proba(X_val)
    for name, search in searches.items():
        model = search.best_estimator_
        p_val = model.predict_proba(X_val)[:, 1]
        if name == "logistic":
            np.testing.assert_allclose(p_val, baseline, rtol=1e-12, atol=1e-12)
        p_train = model.predict_proba(X_train)[:, 1]
        full_train[name] = probability_metrics(y_train, p_train)

        threshold = threshold_metrics(y_val, p_val, 0.5)
        validation[name] = {
            **probability_metrics(y_val, p_val),
            "Precision": threshold["Precisión"],
            "Recall": threshold["Sensibilidad"],
            "F1": threshold["F1"],
        }
        path = save_model(model, reports / "phase5_models" / f"{name}.joblib")
        np.testing.assert_allclose(load_model(path).predict_proba(X_val)[:, 1], p_val)

display(pd.DataFrame(full_train).T.rename(index=model_names))
display(pd.DataFrame(validation).T.rename(index=model_names))


## Importancia de variables

Las importancias internas suman las columnas codificadas de cada variable original. Para árbol y bosque se usa reducción de impureza. Para boosting se usa ganancia total normalizada. La permutación desordena cada variable original en validación y mide la caída de ROC-AUC (12 repeticiones, barras de error de una desviación estándar). Las variables correlacionadas pueden repartirse importancia. Los rankings describen valor predictivo para estos modelos en Taiwán (2005). Su aplicación a Colombia requiere validación independiente y las importancias no identifican efectos causales.

In [ ]:
importance = {
    name: compare_feature_importance(search.best_estimator_, X_val, y_val)
    for name, search in searches.items()
    if name != "logistic"
}
figure, axes = plt.subplots(3, 2, figsize=(12, 10), layout="constrained")
figure.suptitle("Importancia de variables en modelos de incumplimiento", fontsize=17, weight="bold")
for row, (name, table) in enumerate(importance.items()):
    for col, measure in enumerate(("interna", "permutación_media")):
        axis = axes[row, col]
        top = table.nlargest(8, measure).sort_values(measure)
        error = top["permutación_de"] if col else None
        color = ("#285a77", "#087e83")[col]
        axis.barh(top.index, top[measure], xerr=error, color=color, capsize=3)
        axis.set_axisbelow(True)
        axis.grid(axis="x", color="#dce5eb")
        axis.spines[["top", "right", "left"]].set_visible(False)
        axis.tick_params(axis="y", length=0, labelsize=9)
        internal_label = "Impureza normalizada" if name in {"tree", "forest"} else "Ganancia normalizada"
        axis.set_xlabel("Caída de ROC-AUC en validación" if col else internal_label)
        axis.set_title(f"{model_names[name]} · {'permutación' if col else 'importancia interna'}",
                       loc="left", fontsize=11, weight="bold")
        if col:
            axis.axvline(0, color="#526474", linewidth=0.8)
figures = root / "figures"
figures.mkdir(exist_ok=True)
png_path = figures / "07_ml_feature_importance.png"
figure.savefig(png_path, dpi=300, facecolor="white")
plt.close(figure)
with Image.open(png_path) as png:
    assert png.size == (3600, 3000)
    np.testing.assert_allclose(png.info["dpi"], (300, 300), atol=0.01)
display(pd.concat(importance, names=["modelo"]).groupby(level=0)["permutación_media"].nlargest(5))
print(png_path)


In [ ]:
report = {
    "partición": partitions,
    "modelos": model_names,
    "versiones": {package: version(package) for package in (
        "scikit-learn", "pandas", "numpy", "scipy", "statsmodels", "joblib",
        *(("xgboost",) if boosting == "xgboost" else ()),
    )},
    "semilla": 42,
    "umbral": 0.5,
    "permutaciones": 12,
    "CV": cv,
    "train completo": full_train,
    "val": validation,
    "importancia": {name: table.to_dict("index") for name, table in importance.items()},
}
report_path = reports / "phase5_results.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8",
)
print(report_path)
